# Exercise sheet 5

*This assignment was done by group of ID*

<YOUR_GROUP_ID> e.g. 10

<YOUR_GROUP_DAY> Tuesday or Wednesday

## 1. Coding

### a) Strided convolution (1 point)
In the lecture you learned about the convolution operation. It was also introduced as a sliding window operation. The stride, in this intuition, is the step size of the sliding window.
In this exercise, you will implement a 2d convolutional layer with the additional option to use a stride. The strided convoultion $x*k$ for a given 2d input signal $x \in \mathbb{R}^{H \times W}$, a kernel $k  \in \mathbb{R}^{2k+1 \times 2k+1}$ and a stride $s$ is defined as follows:

$$(x * k)(i, j) = \sum_{m=-k}^{k} \sum_{n=-k}^{k} x(is + m, js + n) \cdot k(m, n)$$

The following figure illustrates the strided convolution with a stride of 2:

![Convolution with stride 2](https://upload.wikimedia.org/wikipedia/commons/0/04/Convolution_arithmetic_-_Padding_strides.gif)


Your task is to extend the code given in the lecture to include the stride parameter. The function should have the following signature:

```python
def conv2d(self: Tensor, kernel: Tensor, stride = 1: int, padding = None: int) -> Tensor:
```

In [1]:
import numpy as np
from Tensor import Tensor

In [2]:
def conv2d(self: Tensor, kernel: Tensor, stride: int = 1, padding: int = None) -> Tensor:
    ## your code here ##
    kernel = kernel.data
    tensor = np.pad(self.data, padding or kernel.shape[-1] // 2)

    x_starts = np.arange((tensor.shape[0] - kernel.shape[0]) + 1, step=stride)
    x_ends = x_starts + kernel.shape[0]

    y_starts = np.arange((tensor.shape[1] - kernel.shape[1]) + 1, step=stride)
    y_ends = y_starts + kernel.shape[1]

    out = np.empty((len(x_starts), len(y_starts)))

    for i, x_start, x_end in zip(range(len(x_starts)), x_starts, x_ends):
        for j, y_start, y_end in zip(range(len(y_starts)), y_starts, y_ends):
            out[i, j] = (tensor[x_start:x_end, y_start:y_end] * kernel).sum()

    return Tensor(out)

#### Tests

In [3]:
import imageio.v3 as imageio

url = 'https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png'
lena = imageio.imread(url)

from PIL import Image

# make gray-scale image
lena_bw = Image.fromarray(lena).convert('L')
lena_np = np.array(lena_bw).astype(np.float32) / 255.0
lena_tensor = Tensor(data=lena_np)
smoothing_kernel = np.array([[1, 1, 1], [1, 1, 1], [1, 1, 1]]).astype(np.float32) / 9
smoothing_kernel = Tensor(data=smoothing_kernel)
padding = None

#### Functional tests
Your implementation should pass the following functional tests:

In [4]:
import torch
import torch.nn.functional as F

lena_torch = torch.tensor(lena_np)
smoothing_kernel_torch = torch.tensor(smoothing_kernel.data)

for stride in [1, 2, 4, 8]:
    output_torch = F.conv2d(lena_torch[None, None, ...], smoothing_kernel_torch[None, None, ...],
                            stride=stride,
                            padding=padding if padding is not None else smoothing_kernel.data.shape[
                                                                            -1] // 2)
    output = conv2d(lena_tensor, smoothing_kernel, stride, padding)
    assert output.data.shape == output_torch.squeeze().numpy().shape, f"Shape mismatch: {output.data.shape} != {output_torch.squeeze().numpy().shape}"
    np.testing.assert_allclose(output.data, output_torch.squeeze().numpy(), rtol=1e-5)
    print(f"Stride: {stride} - Correct")

Stride: 1 - Correct
Stride: 2 - Correct
Stride: 4 - Correct
Stride: 8 - Correct


#### Performance test
This test is just for your interest. You can use it to compare different implementations you come up with or against the pytorch implementation.

In [5]:
np.random.seed(42)
data = np.random.rand(1000, 1000).astype(np.float32)
data_tensor = Tensor(data=data)
kernel = np.random.rand(3, 3).astype(np.float32)
kernel_tensor = Tensor(data=kernel)
stride = 2
padding = None

In [6]:
## Your implementation
%timeit conv2d(data_tensor, kernel_tensor, stride, padding)

522 ms ± 23.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
## PyTorch implementation
%timeit F.conv2d(torch.tensor(data[None, None, ...]), torch.tensor(kernel[None, None, ...]), stride=stride, padding=padding if padding is not None else kernel.shape[-1] // 2)

2.48 ms ± 78.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### b) 2d transposed convolution (2 points)

In this exercise, you will implement a 2d transposed convolution operation $x *^T k$ for a given 2d input signal $x \in \mathbb{R}^{H \times W}$ and a kernel $k \in \mathbb{R}^{2k + 1 \times 2k+1}$ with unit stride and no padding.

The 2d transposed convolution can be understood as a sliding window operation over the elements in the input signal $x$. The whole filter $k$ is applied to each element of the input signal $x$ by multiplying the filter with the element under consideration and adding the result to the output tensor at the corresponding position.
To implement the transposed convolution, you can initialize the output tensor with zeros and then iteratevely add the result up.

The following figure illustrates the transposed convolution in terms of an example

![Transposed convolution](https://www.researchgate.net/profile/Md-Afif-Al-Mamun-2/publication/358607641/figure/fig4/AS:1123654414934027@1644911497422/6-Transposed-convolution-operation-6-shows-transposed-convolution-of-a-2-2-input.ppm)


Your task is to implement the transposed convolution operation. The function should have the following signature:

```python
def conv_transpose2d(self: Tensor, kernel: Tensor) -> Tensor:
```

In [8]:
def conv_transpose2d(self: Tensor, kernel: Tensor) -> Tensor:
    ## your code here ##
    kernel = kernel.data
    data = self.data

    x_starts = np.arange(data.shape[0])
    x_ends = x_starts + kernel.shape[0]

    y_starts = np.arange(data.shape[1])
    y_ends = y_starts + kernel.shape[1]

    out = np.zeros((data.shape[0] + kernel.shape[0] - 1, data.shape[1] + kernel.shape[1] - 1))

    for i, x_start, x_end in zip(range(len(x_starts)), x_starts, x_ends):
        for j, y_start, y_end in zip(range(len(y_starts)), y_starts, y_ends):
            out[x_start:x_end, y_start:y_end] += data[i, j] * kernel

    return out

### Tests

#### Functional tests
Your implementation should pass the following functional tests, which test the shape of the output and also the numerical values of it.

In [9]:
np.random.seed(42)

for input_size in [(3, 3), (4, 4), (5, 5), (6, 6)]:
    for kernel_size in [(1, 1), (2, 2), (3, 3), (4, 4)]:
        data = np.random.rand(*input_size).astype(np.float32)
        data_tensor = Tensor(data=data)
        kernel = np.random.rand(*kernel_size).astype(np.float32)
        kernel_tensor = Tensor(data=kernel)

        output_torch = F.conv_transpose2d(torch.tensor(data[None, None, ...]),
                                          torch.tensor(kernel[None, None, ...]))

        output = conv_transpose2d(data_tensor, kernel_tensor)
        assert output.data.shape == output_torch.squeeze().numpy().shape, f"Shape mismatch: {output.data.shape} != {output_torch.squeeze().numpy().shape}"
        np.testing.assert_allclose(output.data, output_torch.squeeze().numpy(), rtol=1e-5)
        print(f"Input size: {input_size}, Kernel size: {kernel_size} - Correct")

Input size: (3, 3), Kernel size: (1, 1) - Correct
Input size: (3, 3), Kernel size: (2, 2) - Correct
Input size: (3, 3), Kernel size: (3, 3) - Correct
Input size: (3, 3), Kernel size: (4, 4) - Correct
Input size: (4, 4), Kernel size: (1, 1) - Correct
Input size: (4, 4), Kernel size: (2, 2) - Correct
Input size: (4, 4), Kernel size: (3, 3) - Correct
Input size: (4, 4), Kernel size: (4, 4) - Correct
Input size: (5, 5), Kernel size: (1, 1) - Correct
Input size: (5, 5), Kernel size: (2, 2) - Correct
Input size: (5, 5), Kernel size: (3, 3) - Correct
Input size: (5, 5), Kernel size: (4, 4) - Correct
Input size: (6, 6), Kernel size: (1, 1) - Correct
Input size: (6, 6), Kernel size: (2, 2) - Correct
Input size: (6, 6), Kernel size: (3, 3) - Correct
Input size: (6, 6), Kernel size: (4, 4) - Correct


#### Performance test

In [10]:
np.random.seed(42)
data = np.random.rand(1000, 1000).astype(np.float32)
data_tensor = Tensor(data=data)
kernel = np.random.rand(11, 11).astype(np.float32)
kernel_tensor = Tensor(data=kernel)

In [11]:
%timeit conv_transpose2d(data_tensor, kernel_tensor)

4.37 s ± 1.69 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%timeit F.conv_transpose2d(torch.tensor(data[None, None, ...]), torch.tensor(kernel[None, None, ...]))

320 ms ± 34.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


--- 

## 2. Theory (2 point)

Derive the formula for the output shape of a convolutional layer. Assume the input shape is $H \times W$ and the convolutional layer has a kernel of dimension $F \times F$. The stride is $S$ and the padding is $P$. (i.e. come up with a formula that gives you $H'$ and $W'$ for the output shape $H' \times W'$ of the convolutional layer in terms of $H$, $W$, $F$, $S$ and $P$ and explain why it is correct).

$H' = \left\lfloor \frac{H + 2P - F}{S} \right\rfloor + 1$

analogue $W'$

The $H + 2P$ is the new total height of the input. \
$\Rightarrow H_P=H + 2P$

If the kernel has a height of $1$ the output trivially has the same height as the input (assuming 
$S=1$). \
By increasing the kernel height by $x$, we reduce the output height by $x$ also, since we can't 
start the pooling from the last $x-1$ rows, as there isn't enough rows below to fit the kernel. \
$\Rightarrow \text{add: } H_F=H_P-F+1$

Assume we applied padding and pooled all with a stride of 1. The resultant matrix has the size 
$H_F$. On that matrix we can very simply calculate any stride, by simply taking the product of 
indices $(0*S, 1*S, \dots, H_F \texttt{ // } S * S)$ in both x and y direction (\texttt{ // } 
being integer division).

Counting those, it's $\frac{H_F}{S} + 1$ if $H_F \equiv 0 (\textrm{mod}\ S)$, and $\frac{H_F}{S}$ 
if $H_F \not\equiv 0 (\textrm{mod}\ S)$, also know as $\left\lfloor \frac{H + 2P - F}{S} 
\right\rfloor + 1$

$$
\begin{align} \Box& \end{align}
$$